# Lesson 9A: Association Rule Learning - Theory

## Market Basket Analysis

**Case Study**: Amazon's "Customers who bought this also bought..." feature drives 35% of revenue. How do they discover these associations?

**Goal**: Find patterns in transactions

**Classic example**: "Customers who buy diapers also buy beer" (found by Walmart data mining)

**Applications**:
- 🛒 **Retail**: Product placement, cross-selling
- 🎬 **Streaming**: Content recommendations
- 📚 **Publishing**: Book bundles
- 🏥 **Healthcare**: Drug interaction patterns

In [ ]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt

np.random.seed(42)
print("✅ Loaded!")

## Key Metrics

For rule A → B:

1. **Support**: P(A ∩ B) = |transactions with A and B| / |total transactions|
   - Measures popularity of itemset
   - Example: support({milk, bread}) = 0.3 means 30% of transactions have both

2. **Confidence**: P(B|A) = P(A ∩ B) / P(A)
   - Measures reliability of rule
   - Example: confidence(milk → bread) = 0.8 means 80% who buy milk also buy bread

3. **Lift**: P(A ∩ B) / (P(A) × P(B))
   - Measures strength of association
   - Lift > 1: Positive correlation
   - Lift = 1: Independent
   - Lift < 1: Negative correlation

**Goal**: Find rules with high support (frequent), high confidence (reliable), high lift (strong)

In [ ]:
# Demonstrate metrics
transactions = [
    ['milk', 'bread', 'butter'],
    ['milk', 'bread', 'beer'],
    ['bread', 'butter'],
    ['milk', 'bread', 'butter', 'cheese'],
    ['bread', 'cheese'],
    ['milk', 'butter'],
    ['bread', 'milk'],
    ['butter', 'cheese'],
]

# Count occurrences
n = len(transactions)
milk_count = sum('milk' in t for t in transactions)
bread_count = sum('bread' in t for t in transactions)
both_count = sum('milk' in t and 'bread' in t for t in transactions)

# Calculate metrics
support = both_count / n
confidence = both_count / milk_count
p_milk = milk_count / n
p_bread = bread_count / n
lift = support / (p_milk * p_bread)

print(f"Rule: milk → bread")
print(f"  Support: {support:.2f} ({both_count}/{n} transactions)")
print(f"  Confidence: {confidence:.2f} ({both_count}/{milk_count} who bought milk)")
print(f"  Lift: {lift:.2f} ({lift:.1f}x more likely than random)")
print("✅ Metrics computed!")

## Apriori Algorithm

**Challenge**: With 1000 items, there are 2^1000 possible itemsets to check!

**Key insight**: **Apriori property** - if an itemset is frequent, all its subsets must be frequent.

**Algorithm**:
1. Find frequent 1-itemsets (single items)
2. Generate candidate 2-itemsets from frequent 1-itemsets
3. Prune candidates using minimum support
4. Repeat for k-itemsets until no more frequent sets found

**Efficiency**: Dramatically reduces search space

In [ ]:
# Using production library
# Encode transactions
te = TransactionEncoder()
te_ary = te.fit_transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("One-hot encoded transactions:")
print(df.head())

# Find frequent itemsets
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)
print(f"\nFrequent Itemsets ({len(frequent_itemsets)}):")
print(frequent_itemsets)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values('lift', ascending=False)

print(f"\nAssociation Rules ({len(rules)}):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))
print("✅ Rules discovered!")

## Visualizing Rules

Understanding the support-confidence-lift relationship

In [ ]:
# Visualize rules
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Support vs Confidence (size = lift)
ax = axes[0]
scatter = ax.scatter(rules['support'], rules['confidence'], 
                    s=rules['lift']*100, alpha=0.6, c=rules['lift'], 
                    cmap='viridis', edgecolors='black', linewidths=0.5)
ax.set_xlabel('Support (Frequency)', fontsize=12)
ax.set_ylabel('Confidence (Reliability)', fontsize=12)
ax.set_title('Association Rules (size & color = lift)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Lift', fontsize=11)

# Lift distribution
ax = axes[1]
ax.hist(rules['lift'], bins=20, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Lift = 1 (independent)')
ax.set_xlabel('Lift', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Lift Distribution', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ Visualization complete!")

## When to Use Association Rules?

✅ **Good for**:
- Transaction data (market baskets, clickstreams)
- Finding co-occurrence patterns
- Recommendation systems (complementary items)
- Sparse binary data

❌ **Not ideal for**:
- Continuous numerical data
- Temporal sequences (use sequential pattern mining)
- Causal relationships (correlation ≠ causation)

**Production tips**:
- Start with min_support = 0.01 to 0.05 (1-5%)
- Use min_confidence = 0.5 to 0.7 (50-70%)
- Filter by lift > 1.5 for strong associations
- Validate rules with domain experts (avoid spurious correlations)